# ACL Tear Detection — V8.3 MRNet Dataset + Anti-Overfit Fixes

**V8.2 problem:** Fully unfrozen backbone → rapid overfitting (14% gap by epoch 11)

**V8.3 first attempt:** Froze 6/8 blocks + 0.7 dropout → total underfitting (model stuck at
majority-class 79%, AUC≈0.50 for 6+ epochs). ImageNet features from frozen blocks are too
generic for grayscale MRI — the model had no capacity to adapt.

**V8.3 fix: moderate regularization that still allows MRI adaptation**

| Setting | V8.2 (overfit) | V8.3 attempt 1 (underfit) | V8.3 final |
|---------|---------------|--------------------------|------------|
| Frozen blocks | 0/8 | 6/8 | **3/8** |
| Dropout | 0.5 | 0.7 | **0.5** |
| Backbone LR | 5e-5 | 1e-5 | **2e-5** |
| Classifier LR | 5e-5 | 5e-5 | **1e-4** |
| Weight decay | 0.01 | 0.02 | **0.01** |
| Max-pool | Unmasked | Masked | **Masked** |
| Warmup | None | 3 epochs (10%→100%) | **3 epochs (10%→100%)** |
| Augmentation | Flip+rot | +brightness/contrast/noise | **+brightness/contrast/noise** |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Configuration
# ============================================================
MRNET_DATA_DIR = '/content/drive/MyDrive/dataset/mrnet_sagittal'
PRIYANK_DATA_DIR = '/content/drive/MyDrive/dataset/combined'

BATCH_SIZE = 1           # MRNet uses batch_size=1 (variable slice count)
NUM_EPOCHS = 50
RANDOM_SEED = 42

MAX_SLICES = 40          # Pad/truncate to this (but masked in max-pool)
SLICE_RANGE_PRIYANK = (33, 43)

# ---- V8.3 Settings (balanced: not too much, not too little) ----
BACKBONE_LR = 2e-5       # Higher than attempt 1 (1e-5) — needs to adapt to MRI
CLASSIFIER_LR = 1e-4     # 5x backbone — classifier head needs faster learning
WEIGHT_DECAY = 0.01      # Back to V8.2 level (0.02 was too aggressive)
DROPOUT = 0.5            # Back to V8.2 level (0.7 killed the signal)
FREEZE_BLOCKS = 3        # Only freeze low-level features (edges/textures)
WARMUP_EPOCHS = 3        # Linear warmup (start at 10% LR, ramp to 100%)
PATIENCE = 10
N_FOLDS = 5

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# ============================================================
# Load Metadata
# ============================================================
mrnet_path = Path(MRNET_DATA_DIR)
mrnet_df = pd.read_csv(mrnet_path / 'metadata.csv')
mrnet_df['label_binary'] = mrnet_df['label'].astype(int)
mrnet_df['label_name_binary'] = mrnet_df['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"MRNet Dataset: {len(mrnet_df)} patients")
print(mrnet_df['label_name_binary'].value_counts())
print(f"\nOriginal split:")
print(pd.crosstab(mrnet_df['split'], mrnet_df['label_name_binary'], margins=True))
print(f"\nSlice count: min={mrnet_df['num_slices'].min()}, max={mrnet_df['num_slices'].max()}, mean={mrnet_df['num_slices'].mean():.1f}")

# Priyank metadata
priyank_path = Path(PRIYANK_DATA_DIR)
all_metadata = pd.read_csv(priyank_path / 'metadata.csv')
priyank_df = all_metadata[all_metadata['source'] == 'Priyank_Saxena'].reset_index(drop=True)
priyank_df['label_binary'] = (priyank_df['label'] > 0).astype(int)
priyank_df['label_name_binary'] = priyank_df['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"\nPriyank (External Validation): {len(priyank_df)} patients")
print(priyank_df['label_name_binary'].value_counts())

In [ ]:
# ============================================================
# Dataset — V8.3: Tracks actual slice count for masked max-pool
#                  + stronger augmentation
# ============================================================

class MRNetVolumeDataset(Dataset):
    """MRNet volume dataset with actual slice tracking and stronger augmentation."""

    def __init__(self, df, data_dir, max_slices=MAX_SLICES, augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.max_slices = max_slices
        self.augment = augment

        # ImageNet normalization for EfficientNet
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        # Verify files exist
        self.valid_indices = []
        for idx, row in self.df.iterrows():
            fpath = self.data_dir / row['filename']
            if fpath.exists():
                self.valid_indices.append(idx)

        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def _augment_volume(self, slices):
        """V8.3: Stronger augmentation — flip, rotation, brightness, contrast, noise."""
        import cv2

        # Random horizontal flip (consistent across all slices)
        if np.random.random() < 0.5:
            slices = slices[:, :, ::-1].copy()

        # Random rotation (consistent across all slices)
        if np.random.random() < 0.5:
            angle = np.random.uniform(-15, 15)
            h, w = slices.shape[1], slices.shape[2]
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            slices = np.stack([
                cv2.warpAffine(s, M, (w, h), borderValue=0) for s in slices
            ])

        # Random brightness shift (consistent across volume)
        if np.random.random() < 0.4:
            brightness = np.random.uniform(-0.1, 0.1)
            slices = np.clip(slices + brightness, 0, 1)

        # Random contrast adjustment (consistent across volume)
        if np.random.random() < 0.4:
            contrast = np.random.uniform(0.8, 1.2)
            mean_val = slices.mean()
            slices = np.clip((slices - mean_val) * contrast + mean_val, 0, 1)

        # Random Gaussian noise
        if np.random.random() < 0.3:
            noise = np.random.normal(0, 0.02, slices.shape).astype(np.float32)
            slices = np.clip(slices + noise, 0, 1)

        return slices

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]

        # Load volume
        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']  # (S, 256, 256), uint8

        # Use ALL slices, normalize to [0,1]
        slices = volume.astype(np.float32) / 255.0

        # Track actual slice count BEFORE padding
        actual_slices = slices.shape[0]

        # Pad or truncate to max_slices
        if actual_slices < self.max_slices:
            pad = np.zeros((self.max_slices - actual_slices, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual_slices > self.max_slices:
            # Center crop
            offset = (actual_slices - self.max_slices) // 2
            slices = slices[offset:offset + self.max_slices]
            actual_slices = self.max_slices  # All slices are real after truncation

        # Augmentation
        if self.augment:
            slices = self._augment_volume(slices)

        # 3-channel + ImageNet normalization
        slices_3ch = np.stack((slices,)*3, axis=1)  # (S, 3, H, W)
        slices_tensor = torch.FloatTensor(slices_3ch)
        slices_tensor = torch.stack([self.normalize(s) for s in slices_tensor])

        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx, actual_slices

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]


class PriyankVolumeDataset(Dataset):
    """Dataset for Priyank .npz volumes (uses fixed slice range)."""

    def __init__(self, df, data_dir, slice_range=SLICE_RANGE_PRIYANK, max_slices=MAX_SLICES):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.slice_range = slice_range
        self.max_slices = max_slices

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

        self.valid_indices = []
        for idx, row in self.df.iterrows():
            fpath = self.data_dir / row['filename']
            if fpath.exists():
                self.valid_indices.append(idx)

        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]

        file_path = self.data_dir / row['filename']
        volume = np.load(file_path)['data']

        start, end = self.slice_range
        start = min(start, volume.shape[0])
        end = min(end, volume.shape[0])
        slices = volume[start:end].astype(np.float32) / 255.0

        actual_slices = slices.shape[0]

        if actual_slices < self.max_slices:
            pad = np.zeros((self.max_slices - actual_slices, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual_slices > self.max_slices:
            slices = slices[:self.max_slices]
            actual_slices = self.max_slices

        slices_3ch = np.stack((slices,)*3, axis=1)
        slices_tensor = torch.FloatTensor(slices_3ch)
        slices_tensor = torch.stack([self.normalize(s) for s in slices_tensor])

        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx, actual_slices

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]

In [ ]:
# ============================================================
# Model — V8.3: Partially frozen backbone + masked max-pool
# ============================================================

class ACLVolumeNet(nn.Module):
    """MRNet architecture with partial EfficientNet-B0 freezing and masked max-pool.
    
    Freeze first 3 blocks (low-level edge/texture features that transfer well)
    and fine-tune blocks 3-7 (higher-level features that need MRI adaptation).
    """
    def __init__(self, dropout=DROPOUT, freeze_blocks=FREEZE_BLOCKS):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = backbone.features  # 9 sub-modules: features[0] through features[8]
        self.pool = backbone.avgpool
        self.in_features = backbone.classifier[1].in_features  # 1280

        # Freeze first N blocks (low-level features)
        for i in range(min(freeze_blocks, len(self.features))):
            for param in self.features[i].parameters():
                param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.in_features, 2)
        )

        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = frozen + trainable
        print(f"  Backbone: EfficientNet-B0, blocks 0-{freeze_blocks-1} FROZEN")
        print(f"  Trainable blocks: {freeze_blocks}-{len(self.features)-1} + classifier")
        print(f"  Classifier: Dropout({dropout}) -> Linear({self.in_features}, 2)")
        print(f"  Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")
        print(f"  Frozen: {frozen:,} ({100*frozen/total:.1f}%)")

    def forward(self, x, actual_slices=None):
        """Forward pass with masked max-pool."""
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        features = self.features(x)
        features = self.pool(features)
        features = features.flatten(1)         # (B*S, 1280)
        features = features.view(B, S, -1)     # (B, S, 1280)

        # Masked max-pool: set padded slices to -inf so they can't win
        if actual_slices is not None:
            mask = torch.arange(S, device=features.device).unsqueeze(0) >= actual_slices.unsqueeze(1)
            mask = mask.unsqueeze(2).expand_as(features)
            features = features.masked_fill(mask, float('-inf'))

        pooled, _ = torch.max(features, dim=1)  # (B, 1280)
        logits = self.classifier(pooled)         # (B, 2)
        return logits

    def get_param_groups(self, backbone_lr, classifier_lr):
        """Differential learning rates for backbone vs classifier."""
        backbone_params = []
        classifier_params = []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if 'classifier' in name:
                classifier_params.append(param)
            else:
                backbone_params.append(param)
        return [
            {'params': backbone_params, 'lr': backbone_lr},
            {'params': classifier_params, 'lr': classifier_lr},
        ]

In [ ]:
# ============================================================
# Warmup + Cosine Annealing Scheduler
# ============================================================

def create_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    """Linear warmup (10% -> 100%) then cosine annealing."""
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch
    min_factor = 0.1  # Start at 10% of target LR

    def warmup_fn(step):
        if step < warmup_steps:
            return min_factor + (1.0 - min_factor) * float(step) / float(max(1, warmup_steps))
        return 1.0

    warmup_scheduler = LambdaLR(optimizer, lr_lambda=warmup_fn)
    cosine_scheduler = CosineAnnealingLR(
        optimizer, T_max=total_steps - warmup_steps, eta_min=1e-7
    )
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )
    return scheduler

In [ ]:
# ============================================================
# Training & Validation
# ============================================================

def train_epoch(model, loader, criterion, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    for volumes, labels, _, actual_slices in tqdm(loader, desc='Training', leave=False):
        volumes = volumes.to(device)
        labels = labels.to(device)
        actual_slices = actual_slices.to(device)

        optimizer.zero_grad()
        logits = model(volumes, actual_slices=actual_slices)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for volumes, labels, _, actual_slices in tqdm(loader, desc='Validating', leave=False):
            volumes = volumes.to(device)
            labels = labels.to(device)
            actual_slices = actual_slices.to(device)

            logits = model(volumes, actual_slices=actual_slices)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc, all_preds, all_labels

---
## Part A: Single-Split Training (MRNet → Priyank External Validation)

In [ ]:
# ============================================================
# Train/Val/Test Split — MRNet Data (70/15/15)
# ============================================================
train_df, temp_df = train_test_split(
    mrnet_df, test_size=0.3, stratify=mrnet_df['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print("MRNet Dataset Split:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_name_binary'].value_counts().to_dict()
    print(f"  {name}: {len(df)} patients — {dist}")

print(f"\nExternal (Priyank): {len(priyank_df)} patients — {priyank_df['label_name_binary'].value_counts().to_dict()}")

In [ ]:
# ============================================================
# Create Datasets & DataLoaders
# ============================================================
print("Creating datasets...")
print(f"  MAX_SLICES={MAX_SLICES} (pad/truncate, but masked in max-pool)")
print()

print("Train (MRNet, augmented):")
train_dataset = MRNetVolumeDataset(train_df, MRNET_DATA_DIR, augment=True)
print("Val (MRNet):")
val_dataset = MRNetVolumeDataset(val_df, MRNET_DATA_DIR, augment=False)
print("Test (MRNet):")
test_dataset = MRNetVolumeDataset(test_df, MRNET_DATA_DIR, augment=False)
print("Priyank (external):")
priyank_dataset = PriyankVolumeDataset(priyank_df, PRIYANK_DATA_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
priyank_loader = DataLoader(priyank_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain: {len(train_dataset)} patients, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} patients")
print(f"Test:  {len(test_dataset)} patients")
print(f"Priyank: {len(priyank_dataset)} patients")

In [ ]:
# ============================================================
# Model, Loss, Optimizer
# ============================================================
print("Creating V8.3 model...")
model = ACLVolumeNet()
model = model.to(device)

# Class weights
train_labels = train_dataset.get_labels()
n_total = len(train_labels)
n_pos = sum(train_labels)
n_neg = n_total - n_pos
w_normal = n_total / (2 * n_neg)
w_tear = n_total / (2 * n_pos)
class_weights = torch.FloatTensor([w_normal, w_tear]).to(device)
print(f"  Class counts: Normal={n_neg}, Tear={n_pos}")
print(f"  Class weights: Normal={w_normal:.3f}, Tear={w_tear:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Differential learning rates
param_groups = model.get_param_groups(BACKBONE_LR, CLASSIFIER_LR)
optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Warmup + Cosine Annealing
scheduler = create_scheduler(
    optimizer,
    warmup_epochs=WARMUP_EPOCHS,
    total_epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader)
)

print(f"\nV8.3 Settings:")
print(f"  Loss:          CrossEntropyLoss (class-weighted)")
print(f"  Optimizer:     AdamW (backbone_lr={BACKBONE_LR}, classifier_lr={CLASSIFIER_LR}, wd={WEIGHT_DECAY})")
print(f"  Scheduler:     Linear warmup ({WARMUP_EPOCHS} epochs, 10%->100%) + Cosine Annealing")
print(f"  Backbone:      Blocks 0-{FREEZE_BLOCKS-1} frozen, {FREEZE_BLOCKS}-8 trainable")
print(f"  Dropout:       {DROPOUT}")
print(f"  Max-pool:      MASKED (padded slices set to -inf)")
print(f"  Augmentation:  Flip + rotation + brightness + contrast + noise")
print(f"  Early stop:    patience={PATIENCE} on val_AUC")

In [ ]:
# ============================================================
# Training Loop
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'train_auc': [],
           'val_loss': [], 'val_acc': [], 'val_auc': [], 'lr': []}
best_val_auc = 0.0
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v8.3.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE} on val_AUC)...\n")

for epoch in range(NUM_EPOCHS):
    current_lr_bb = optimizer.param_groups[0]['lr']
    current_lr_cl = optimizer.param_groups[1]['lr']

    train_loss, train_acc, train_auc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )
    val_loss, val_acc, val_auc, val_preds, val_labels = validate(
        model, val_loader, criterion, device
    )

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['lr'].append(current_lr_bb)

    gap = train_acc - val_acc
    gap_warn = ' OVERFIT' if gap > 10 else ''

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  (bb_lr={current_lr_bb:.1e}, cl_lr={current_lr_cl:.1e})")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%  AUC={train_auc:.4f}")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  AUC={val_auc:.4f}  (gap={gap:.1f}%){gap_warn}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> Saved best (AUC={val_auc:.4f}, acc={val_acc:.2f}%, loss={val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nBest val AUC: {best_val_auc:.4f}")

In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(30, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(history['train_auc'], label='Train')
axes[2].plot(history['val_auc'], label='Val')
axes[2].set_title('AUC'); axes[2].legend(); axes[2].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[3].plot(gaps, color='red')
axes[3].axhline(y=10, color='orange', linestyle='--', label='10% threshold')
axes[3].set_title('Overfit Gap'); axes[3].legend(); axes[3].grid(True)

axes[4].plot(history['lr'], color='purple')
axes[4].set_title('Learning Rate (backbone)'); axes[4].grid(True)
axes[4].set_yscale('log')

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v8.3.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Evaluate on MRNet Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))

test_loss, test_acc, test_auc, test_preds, test_labels = validate(
    model, test_loader, criterion, device
)

label_names = ['Normal', 'Tear']

print('=' * 60)
print('TEST RESULTS — V8.3 (MRNet Data)')
print('=' * 60)
print(f'Accuracy: {test_acc:.2f}%')
print(f'AUC: {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — MRNet Test (V8.3)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8.3_mrnet.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# External Validation — Priyank Saxena
# ============================================================
print('=' * 60)
print('EXTERNAL VALIDATION — Priyank Saxena')
print('=' * 60)

eval_criterion = nn.CrossEntropyLoss()
priyank_loss, priyank_acc, priyank_auc, priyank_preds, priyank_labels = validate(
    model, priyank_loader, eval_criterion, device
)

print(f'Accuracy: {priyank_acc:.2f}%')
print(f'AUC: {priyank_auc:.4f}')
print('\nClassification Report:')
print(classification_report(priyank_labels, priyank_preds, target_names=label_names, digits=3))

cm_p = confusion_matrix(priyank_labels, priyank_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_p, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — Priyank External (V8.3)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8.3_priyank.png', dpi=150)
plt.show()

print(f"\n{'='*60}")
print(f"V8.3 SUMMARY")
print(f"{'='*60}")
print(f"  MRNet Test: {test_acc:.1f}% acc, AUC={test_auc:.4f}")
print(f"  Priyank:    {priyank_acc:.1f}% acc, AUC={priyank_auc:.4f}")

---
## Part B: 5-Fold Stratified CV (MRNet) + Priyank External

In [ ]:
# ============================================================
# K-Fold CV
# ============================================================
print(f"\n{'='*60}")
print(f"{N_FOLDS}-FOLD STRATIFIED CV (MRNet) + Priyank External")
print(f"V8.3")
print(f"{'='*60}\n")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_results = []
all_fold_preds = []
all_fold_labels = []
all_fold_priyank_preds = []
all_fold_priyank_labels = []

print("Priyank dataset:")
priyank_cv_dataset = PriyankVolumeDataset(priyank_df, PRIYANK_DATA_DIR)
priyank_cv_loader = DataLoader(
    priyank_cv_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

label_names = ['Normal', 'Tear']

for fold, (train_idx, val_idx) in enumerate(skf.split(mrnet_df, mrnet_df['label_binary'])):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    fold_train_df = mrnet_df.iloc[train_idx]
    fold_val_df = mrnet_df.iloc[val_idx]

    print(f"  Train: {len(fold_train_df)} — {fold_train_df['label_name_binary'].value_counts().to_dict()}")
    print(f"  Val:   {len(fold_val_df)} — {fold_val_df['label_name_binary'].value_counts().to_dict()}")

    print("  Train dataset:")
    fold_train_ds = MRNetVolumeDataset(fold_train_df, MRNET_DATA_DIR, augment=True)
    print("  Val dataset:")
    fold_val_ds = MRNetVolumeDataset(fold_val_df, MRNET_DATA_DIR, augment=False)

    fold_train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    fold_val_loader = DataLoader(fold_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print("  Creating model...")
    fold_model = ACLVolumeNet()
    fold_model = fold_model.to(device)

    fold_labels = fold_train_ds.get_labels()
    fold_n_pos = sum(fold_labels)
    fold_n_neg = len(fold_labels) - fold_n_pos
    fold_w = torch.FloatTensor([
        len(fold_labels) / (2 * fold_n_neg),
        len(fold_labels) / (2 * fold_n_pos)
    ]).to(device)
    fold_criterion = nn.CrossEntropyLoss(weight=fold_w)

    fold_param_groups = fold_model.get_param_groups(BACKBONE_LR, CLASSIFIER_LR)
    fold_optimizer = optim.AdamW(fold_param_groups, weight_decay=WEIGHT_DECAY)
    fold_scheduler = create_scheduler(
        fold_optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=NUM_EPOCHS,
        steps_per_epoch=len(fold_train_loader)
    )

    best_fold_auc = 0.0
    fold_patience = 0
    best_fold_state = None

    for epoch in range(NUM_EPOCHS):
        t_loss, t_acc, t_auc = train_epoch(
            fold_model, fold_train_loader, fold_criterion, fold_optimizer, fold_scheduler, device
        )
        v_loss, v_acc, v_auc, _, _ = validate(
            fold_model, fold_val_loader, fold_criterion, device
        )

        gap = t_acc - v_acc
        if (epoch + 1) % 5 == 0 or v_auc > best_fold_auc:
            gap_warn = ' OVERFIT' if gap > 10 else ''
            print(f"  Epoch {epoch+1}: train_acc={t_acc:.1f}% val_acc={v_acc:.1f}% val_AUC={v_auc:.4f} gap={gap:.1f}%{gap_warn}")

        if v_auc > best_fold_auc:
            best_fold_auc = v_auc
            best_fold_state = copy.deepcopy(fold_model.state_dict())
            fold_patience = 0
        else:
            fold_patience += 1
            if fold_patience >= PATIENCE:
                print(f"  Early stop at epoch {epoch+1} (best AUC: {best_fold_auc:.4f})")
                break

    fold_model.load_state_dict(best_fold_state)

    _, fold_acc, fold_auc, fold_preds, fold_labels_list = validate(
        fold_model, fold_val_loader, fold_criterion, device
    )
    print(f"\n  MRNet Val: acc={fold_acc:.2f}%, AUC={fold_auc:.4f}")
    print(classification_report(fold_labels_list, fold_preds, target_names=label_names, digits=3, zero_division=0))

    eval_crit = nn.CrossEntropyLoss()
    _, p_acc, p_auc, p_preds, p_labels = validate(fold_model, priyank_cv_loader, eval_crit, device)
    print(f"  Priyank: acc={p_acc:.2f}%, AUC={p_auc:.4f}")
    print(classification_report(p_labels, p_preds, target_names=label_names, digits=3, zero_division=0))

    fold_results.append({
        'fold': fold + 1, 'mri_acc': fold_acc, 'mri_auc': fold_auc,
        'priyank_acc': p_acc, 'priyank_auc': p_auc,
        'best_epoch': epoch + 1 - PATIENCE if fold_patience >= PATIENCE else epoch + 1,
    })

    all_fold_preds.extend(fold_preds)
    all_fold_labels.extend(fold_labels_list)
    all_fold_priyank_preds.extend(p_preds)
    all_fold_priyank_labels.extend(p_labels)

    del fold_model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# ============================================================
# K-Fold Summary
# ============================================================
print(f"\n{'='*60}")
print(f"K-FOLD CV SUMMARY — V8.3")
print(f"{'='*60}")

results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

print(f"\n{'─'*50}")
print(f"MRNet (Same Source):")
print(f"  Mean Accuracy: {results_df['mri_acc'].mean():.2f}% ± {results_df['mri_acc'].std():.2f}%")
print(f"  Mean AUC:      {results_df['mri_auc'].mean():.4f} ± {results_df['mri_auc'].std():.4f}")

print(f"\nPriyank Saxena (External):")
print(f"  Mean Accuracy: {results_df['priyank_acc'].mean():.2f}% ± {results_df['priyank_acc'].std():.2f}%")
print(f"  Mean AUC:      {results_df['priyank_auc'].mean():.4f} ± {results_df['priyank_auc'].std():.4f}")

print(f"\n{'─'*50}")
print("Overall MRNet Patient-Level Report (all folds combined):")
print(classification_report(
    all_fold_labels, all_fold_preds,
    target_names=label_names, digits=3, zero_division=0
))

print("Overall Priyank Patient-Level Report (all folds combined):")
print(classification_report(
    all_fold_priyank_labels, all_fold_priyank_preds,
    target_names=label_names, digits=3, zero_division=0
))

In [ ]:
# ============================================================
# K-Fold Visualization
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

x = results_df['fold']
width = 0.3

axes[0].bar(x - width/2, results_df['mri_acc'], width=width, label='MRNet', color='steelblue')
axes[0].bar(x + width/2, results_df['priyank_acc'], width=width, label='Priyank', color='coral')
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Patient-Level Accuracy per Fold (V8.3)')
axes[0].legend(fontsize=8); axes[0].set_xticks(x); axes[0].grid(axis='y')

axes[1].bar(x - width/2, results_df['mri_auc'], width=width, label='MRNet', color='steelblue')
axes[1].bar(x + width/2, results_df['priyank_auc'], width=width, label='Priyank', color='coral')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('AUC')
axes[1].set_title('Patient-Level AUC per Fold (V8.3)')
axes[1].legend(fontsize=8); axes[1].set_xticks(x); axes[1].grid(axis='y')

cm_mri_all = confusion_matrix(all_fold_labels, all_fold_preds)
sns.heatmap(cm_mri_all, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[2])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].set_title(f'MRNet — {N_FOLDS}-Fold CV Patient-Level CM')

cm_priyank_all = confusion_matrix(all_fold_priyank_labels, all_fold_priyank_preds)
sns.heatmap(cm_priyank_all, annot=True, fmt='d', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[3])
axes[3].set_xlabel('Predicted'); axes[3].set_ylabel('Actual')
axes[3].set_title(f'Priyank External — {N_FOLDS}-Fold Patient-Level CM')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/kfold_results_v8.3.png', dpi=150)
plt.show()